In [1]:
# 대화 이력 요약 및 트리밍
## 메시지 트리밍
from langchain_core.messages import trim_messages
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI
from operator import itemgetter

In [2]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate

chat_history = InMemoryChatMessageHistory()

# 시스템 메시지와 대화 이력을 사용하는 프롬프트 템플릿 정의
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 금융 상담사입니다. 모든 질문에 최선을 다해 답변하십시오."),
        ("placeholder","{chat_history}"), # 대화 이력 추가
        ("human", "{input}"),
    ]
)

In [3]:
# 메시지 트리밍 유틸 설정
trimmer = trim_messages(strategy="last", max_tokens=2, token_counter=len)

In [4]:
model = ChatOpenAI( 
    model="unsloth/gemma-4-e2b-it",
    # model="qwen3-30b-a3b-instruct-2507",
    base_url="http://...:.../v1",
    api_key="lm-studio",
)

In [5]:
# 대화 이력을 관리할 체인 설정
chain = prompt | model

In [6]:
from langchain_core.runnables.history import RunnableWithMessageHistory

# RunnableWithMessageHistory 클래스를 사용해 체인을 감쌉니다.
chain_with_message_history = RunnableWithMessageHistory(
    chain,
    lambda session_id: chat_history, # 세션 ID에 따라 대화 이력을 불러오는 함수
    input_messages_key="input", # 입력 메시지의 키 설정
    history_messages_key="chat_history", # 대화 이력의 키 설정
)

/opt/venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [7]:
# 질문 메시지 체인 실행
chain_with_message_history.invoke(
    {"input": "저축을 늘리기 위해 무엇을 할 수 있나요?"},
    {"configurable": {"session_id":"unused"}},
).content

"저축을 늘리는 것은 재정적 목표를 달성하는 데 매우 중요합니다. 꾸준함과 전략이 결합되면 충분히 가능합니다! 제가 금융 상담사로서, 저축액을 효과적으로 늘릴 수 있는 구체적이고 실용적인 방법들을 단계별로 안내해 드리겠습니다.\n\n---\n\n## 1단계: 현재 재정 상태 파악하기 (기반 다지기)\n\n저축을 늘리기 전에, **돈이 어디로 가고 있는지** 정확히 아는 것이 가장 중요합니다.\n\n1. **수입과 지출 기록:** 지난 한두 달간의 모든 수입(월급, 부수입 등)과 지출 내역(고정비, 변동비 모두 포함)을 꼼꼼하게 기록해 보세요. 가계부 앱이나 엑셀을 활용하면 편리합니다.\n2. **예산 설정:** 기록을 바탕으로 '필수 지출', '선택적 지출'을 구분하고, 저축 목표액을 정하여 예산을 세우세요. (예: 수입의 50%는 고정 지출, 30%는 저축/투자, 20%는 변동 지출)\n3. **재정 목표 설정:** '왜 저축하는가?'에 대한 명확한 목표가 있어야 동기 부여가 됩니다. (예: 1년 안에 비상금 300만 원 모으기, 5년 뒤 주택 계약금 마련 등)\n\n## 2단계: 지출 통제 및 절약 (돈을 아끼는 기술)\n\n저축액을 늘리는 가장 직접적인 방법은 **지출을 줄이는 것**입니다.\n\n* **고정 지출 점검:** 구독 서비스(OTT, 앱 등), 불필요한 보험료, 통신비 등을 정기적으로 검토하여 해지하거나 더 저렴한 상품으로 변경하세요.\n* **변동 지출 관리 (생활 습관 개선):**\n    * **식비 절약:** 외식 횟수를 줄이고 도시락을 싸거나 식단을 계획하여 불필요한 지출을 막습니다.\n    * **충동구매 방지:** 물건을 사기 전에 '정말 필요한가?', '이것이 나의 재정 목표에 도움이 되는가?'를 자문하는 습관을 들입니다. (24시간 규칙 적용)\n* **'선 저축, 후 소비' 원칙:** 월급이 들어오면 가장 먼저 저축할 금액을 별도의 계좌로 이체하세요. 남은 돈으로 생활하면 자연스럽게 지출을 통제하게 됩니다.\n\n## 3단

In [8]:
# 새로운 입력 메시지를 추가하고 체인 실행
chain_with_message_history.invoke(
    {"input": "내가 방금 뭐라고 했나요?"},     # 사용자의 질문
    {"configurable": {"session_id":"unused"}} # 세션 ID 설정
).content

'방금 고객님께서는 **"저축을 늘리기 위해 무엇을 할 수 있나요?"** 라고 질문하셨습니다.\n\n저는 그 질문에 대해 저축을 늘리기 위한 구체적인 5단계 전략(재정 상태 파악, 지출 통제, 수입 증대, 자동화, 투자 시작)과 당장 실천할 수 있는 액션 플랜을 안내해 드렸습니다.'

In [9]:
# 트리밍된 대화 이력과 함께 체인 실행
chain_with_trimming = (
    RunnablePassthrough.assign(chat_history=itemgetter("chat_history") | trimmer)
    | prompt
    | model
)

In [10]:
# 트리밍된 대화 이력을 사용하는 체인 설정
chain_with_trimmed_history = RunnableWithMessageHistory(
    chain_with_trimming,
    lambda session_id: chat_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

/opt/venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [11]:
# 새로운 대화를 추가하고 체인 실행
chain_with_trimmed_history.invoke(
    {"input": "저는 5년 내에 집을 사기 위해 어떤 재정 계획을 세워야하나요?"}, # 사용자의 질문
    {"configurable": {"session_id": "finance_session_1"}} # 세션 ID 설정
)

AIMessage(content="5년 안에 주택 구매라는 명확한 목표를 가지고 계시군요! 이는 매우 구체적이고 달성 가능한 목표이므로, 체계적인 재정 계획을 세우면 충분히 가능합니다.\n\n주택 구매를 위한 5년 재정 계획은 크게 **'목표 설정 및 현황 파악', '자금 마련 전략', '위험 관리 및 실행 계획'**의 세 단계로 나누어 진행해야 합니다.\n\n제가 금융 상담사로서 고객님을 위해 구체적으로 점검하고 준비해야 할 단계를 안내해 드리겠습니다.\n\n---\n\n### 1단계: 현재 재정 상태 정확히 파악하기 (Foundation)\n\n가장 먼저, 현재 위치를 정확히 알아야 목표 지점까지의 거리를 측정할 수 있습니다.\n\n1. **순자산 및 부채 현황 분석:**\n    * **총 자산:** 현재 보유한 모든 자산(예금, 투자금, 주식 등)을 파악합니다.\n    * **부채 목록:** 대출(신용카드, 학자금, 기존 주택 담보대출 등)의 종류, 잔액, 월 상환액을 정확히 기록합니다.\n2. **수입 및 지출 분석 (현금 흐름):**\n    * **월 수입:** 모든 소득원을 합산합니다.\n    * **월 지출:** 고정 지출(주거비, 공과금, 보험료)과 변동 지출(식비, 여가, 쇼핑 등)을 꼼꼼히 추적하여 '실제 저축 가능 금액'을 계산해야 합니다.\n3. **신용 점수 확인:**\n    * 주택 담보 대출 시 유리한 금리를 받기 위해 현재 신용 점수를 확인하고 개선할 부분을 파악합니다.\n\n### 2단계: 목표 설정 및 자금 계획 수립 (Strategy)\n\n5년 후 필요한 '총 필요 자금'을 계산하고, 이를 달성하기 위한 구체적인 로드맵을 만듭니다.\n\n1. **목표 주택 가격 설정:**\n    * 희망하는 지역과 규모를 고려하여 5년 후 예상되는 주택 가격 범위를 현실적으로 설정합니다. (예: 5년 후 4억 원짜리 아파트)\n2. **필요 자금 계산:**\n    * **계약금/초기 비용:** 주택 가격의 일정 비율(보통 10~

In [12]:
# 새로운 입력 메시지를 추가하고 체인 실행
chain_with_trimmed_history.invoke(
    {"input": "내가 방금 뭐라고 했나요?"}, # 사용자의 질문
    {"configurable": {"session_id": "finance_session_1"}} # 세션 ID 설정
).content

'고객님께서는 **"저는 5년 내에 집을 사기 위해 어떤 재정 계획을 세워야하나요?"** 라고 질문하셨습니다.\n\n제가 그 질문에 대해 주택 구매를 위한 3단계 재정 계획(현황 파악, 자금 전략, 실행 및 위험 관리)을 안내해 드렸습니다.'

In [13]:
# 대화 요약 활용
def summarize_messages(chain_input):
    stored_messages = chat_history.messages
    if len(stored_messages) == 0:
        return False

    # 대화를 요약하기 위한 프롬프트 템플릿 설정
    summarization_prompt = ChatPromptTemplate.from_messages(
        [
            ("placeholder", "{chat_history}"), # 이전 대화 이력
            (
                "user",
                "이전 대화를 요약해 주세요. 가능한 한 많은 세부 정보를 포함하십시오.", # 요약 요청 메시지
            )
        ]
    )

    # 요약 체인 생성 및 실행
    summarization_chain = summarization_prompt | model
    summary_message = summarization_chain.invoke({"chat_history": stored_messages})

    chat_history.clear() # 요약 후 이전 대화 삭제
    chat_history.add_message(summary_message) # 요약된 메시지를 대화 이력에 추가

    return True

In [14]:
# 대화 요약을 처리하는 체인 설정
chain_with_summarization = (
    RunnablePassthrough.assign(messages_summarized=summarize_messages)
    | chain_with_message_history
)

In [15]:
# 요약된 대화를 기반으로 새로운 질문에 응답
print(chain_with_summarization.invoke(
    {"input": "저에게 어떤 재정적 조언을 해주셨나요?"},  # 사용자의 질문
    {"configurable": {"session_id": "unused"}}          # 세션 ID 설정
).content)

제가 고객님께 드린 재정적 조언은 크게 **'기초 다지기(습관 형성)'**와 **'목표 달성을 위한 체계적인 로드맵 구축'**이라는 두 가지 축으로 구성되었습니다.

고객님의 상황에 맞춰 구체적으로 드렸던 핵심 조언들을 다시 정리해 드리겠습니다.

### 1. 기초 다지기: 재정 습관 형성 (저축의 시작)

가장 먼저, 돈을 모으는 '습관'을 만드는 것이 중요하다고 강조했습니다.

* **재정 상태 명확히 파악하기:**
    * **수입과 지출 기록:** 현재 내가 얼마를 벌고, 어디에 돈을 쓰는지 정확하게 파악해야 합니다. (가계부 작성 또는 앱 활용)
    * **예산 설정:** 수입에서 고정 지출과 변동 지출을 구분하여 현실적인 예산을 세우는 것이 필수입니다.
* **지출 통제 및 절약 원칙 적용:**
    * **'선 저축 후 소비' 원칙:** 월급이 들어오면 목표 저축액을 먼저 떼어놓고, 남은 돈으로 생활하는 습관을 들여야 합니다.
    * **충동구매 방지:** 불필요한 지출(특히 구독 서비스나 충동적인 쇼핑)을 줄이는 것이 저축액을 늘리는 가장 빠른 길입니다.
* **수입 증대 모색:**
    * 현재 수입만으로는 목표 달성이 어려울 수 있으므로, 자기 계발이나 부업 등을 통해 소득 파이프라인을 다각화하는 방안을 함께 고민해야 한다고 조언했습니다.
* **저축 자동화:**
    * 저축을 '의지'에만 맡기지 말고, 급여일에 맞춰 저축 계좌로 **자동 이체**를 설정하여 습관적으로 돈이 모이도록 시스템을 구축해야 합니다.

### 2. 목표 달성을 위한 체계적인 로드맵 (주택 구매 계획)

단순히 돈을 모으는 것을 넘어, '5년 내 주택 구매'라는 구체적인 목표를 달성하기 위한 단계별 전략을 제시했습니다.

* **1단계: 현재 재정 상태 진단 (Foundation)**
    * **순자산 및 부채 분석:** 현재 내가 가진 자산과 빚(부채)이 어느 정도인지 정확히 알아야 출발점을 알 수 있습니다.
    * **실질 저축 가능 금액 계